# CashGuard — дообучение Qwen2.5-VL-7B (QLoRA)

**Перед запуском:** положи `sharegpt_train.json` и `sharegpt_val.json` в Google Drive → папку `MyDrive/cashguard/`.
Если Drive не монтируется — перетащи оба файла в корень `/content` через панель Files слева.

Запускай ячейки **по порядку сверху вниз**. Любую можно перезапускать — ничего не сломается.
Если после установки (шаг 2) колаб попросит перезапустить сессию — перезапусти и продолжай с шага 3.

## 1. GPU

In [ ]:
import torch
GPU = torch.cuda.get_device_name(0)
print(GPU, "| bf16:", torch.cuda.is_bf16_supported())
if "T4" in GPU:
    print("⚠️ T4: обучение займёт много часов. Если есть Colab Pro — Среда выполнения → Сменить тип среды → A100.")

## 2. Установка LLaMA-Factory

In [ ]:
import os
%cd /content
if not os.path.exists("/content/LLaMA-Factory"):
    !git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd /content/LLaMA-Factory
!pip install -q -e ".[torch,metrics]"
!pip install -q bitsandbytes qwen-vl-utils
print("✔ Установка готова")

## 3. Drive + поиск исходных JSON

In [ ]:
import os, shutil, json
from google.colab import drive

try:
    drive.mount("/content/drive", force_remount=True)
except Exception as e:
    print("⚠️ Drive не подключился:", e)

SRC = "/content/drive/MyDrive/cashguard"
RAW = "/content/cashguard_raw"
os.makedirs(RAW, exist_ok=True)
print("Вижу в MyDrive/cashguard:", os.listdir(SRC) if os.path.isdir(SRC) else "❌ папки нет")

for name in ["sharegpt_train.json", "sharegpt_val.json"]:
    src = f"{SRC}/{name}"
    if not os.path.exists(src):
        src = f"/content/{name}"          # запасной путь: загрузка через панель Files
    if not os.path.exists(src):
        print(f"❌ {name} не найден ни на Drive, ни в /content")
        continue
    shutil.copy(src, f"{RAW}/{name}")
    data = json.load(open(f"{RAW}/{name}", encoding="utf-8"))
    print(f"✔ {name}: {len(data)} сэмплов, {os.path.getsize(src)/1e6:.0f} МБ")

## 4. Конвертация в формат LLaMA-Factory
Вытаскивает кадры из base64 в `.jpg`, ставит `<image>` токены, добавляет поле `images`.

In [ ]:
import os, json, glob, shutil, ast, base64

DATA_DIR   = "/content/LLaMA-Factory/data"
RAW_DIR    = "/content/cashguard_raw"
FRAMES_DIR = os.path.join(DATA_DIR, "cashguard_frames")

def to_parts(value):
    if isinstance(value, list):
        return value
    if isinstance(value, str) and value.lstrip().startswith("[{"):
        try:
            return ast.literal_eval(value)
        except Exception:
            return None
    return None

def convert(split):
    with open(f"{RAW_DIR}/sharegpt_{split}.json", encoding="utf-8") as f:
        data = json.load(f)
    split_dir = os.path.join(FRAMES_DIR, split)
    shutil.rmtree(split_dir, ignore_errors=True)
    os.makedirs(split_dir)

    out, n_imgs = [], 0
    for i, sample in enumerate(data):
        images, convs = [], []
        for turn in sample["conversations"]:
            parts = to_parts(turn["value"])
            if parts is None:
                convs.append({"from": turn["from"], "value": str(turn["value"]).strip()})
                continue
            chunks = []
            for p in parts:
                if p.get("type") == "text":
                    chunks.append(p["text"].strip())
                elif p.get("type") == "image_url":
                    iu = p["image_url"]
                    url = iu["url"] if isinstance(iu, dict) else iu
                    rel = f"cashguard_frames/{split}/{i:04d}_{len(images):02d}.jpg"
                    with open(os.path.join(DATA_DIR, rel), "wb") as img:
                        img.write(base64.b64decode(url.split(",", 1)[1]))
                    images.append(rel)
                    chunks.append("<image>")
            convs.append({"from": turn["from"], "value": "\n".join(c for c in chunks if c)})
        n_imgs += len(images)
        out.append({"conversations": convs, "images": images})

    with open(f"{DATA_DIR}/cashguard_{split}.json", "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False)
    print(f"✔ {split}: {len(out)} сэмплов, {n_imgs} кадров")

convert("train")
convert("val")

# убираем мусор от прошлых попыток
for junk in glob.glob(f"{DATA_DIR}/sharegpt_*.json"):
    os.remove(junk)
shutil.rmtree(os.path.join(DATA_DIR, "frames"), ignore_errors=True)
print("✔ старые sharegpt_*.json и frames/ удалены")

In [ ]:
# 4b. Таймкод из описания → timecode_sec, нормативка вырезается (её подставляет бэкенд из rules_kb.json)
# ВАЖНО: после каждого перезапуска шага 4 запускать и эту ячейку.
import json, re

DATA_DIR = "/content/LLaMA-Factory/data"
TC  = re.compile(r"(\d+(?:[.,]\d+)?)\s*(?:-?й|-?ой)?\s*(?:сек|с\b)", re.I)
SRC = re.compile(r"\s*Нормативн\w* (?:источник|основани\w*|акт)\s*:.*$", re.S | re.I)

for split in ["train", "val"]:
    path = f"{DATA_DIR}/cashguard_{split}.json"
    data = json.load(open(path, encoding="utf-8"))
    fixed = not_found = 0
    for x in data:
        turn = [t for t in x["conversations"] if t["from"] == "gpt"][-1]
        ans = json.loads(turn["value"])
        for v in ans.get("violations", []):
            v["description"] = SRC.sub("", v.get("description", "")).strip()
            m = TC.search(v["description"])
            if m:
                v["timecode_sec"] = float(m.group(1).replace(",", "."))
                fixed += 1
            else:
                not_found += 1
        turn["value"] = json.dumps(ans, ensure_ascii=False)
    json.dump(data, open(path, "w", encoding="utf-8"), ensure_ascii=False)
    print(f"✔ {split}: таймкодов исправлено {fixed}, не найдено {not_found}")

## 5. Проверка датасета
Если тут ❌ — дальше не идём.

In [ ]:
import os, json
from collections import Counter
from PIL import Image

DATA_DIR = "/content/LLaMA-Factory/data"
TOKENS_PER_IMG = 262144 // (28 * 28)   # при image_max_pixels=262144

ok = True
for split in ["train", "val"]:
    with open(f"{DATA_DIR}/cashguard_{split}.json", encoding="utf-8") as f:
        data = json.load(f)

    mismatch = sum(1 for x in data
                   if sum(t["value"].count("<image>") for t in x["conversations"]) != len(x["images"]))
    no_imgs  = sum(1 for x in data if not x["images"])
    missing  = sum(1 for x in data for p in x["images"] if not os.path.exists(os.path.join(DATA_DIR, p)))
    max_imgs = max(len(x["images"]) for x in data)

    positives, events, zero_tc, bad_json = 0, 0, 0, 0
    types = Counter()
    for x in data:
        answer = [t["value"] for t in x["conversations"] if t["from"] == "gpt"][-1]
        try:
            viol = json.loads(answer).get("violations", [])
        except Exception:
            bad_json += 1
            continue
        positives += bool(viol)
        for v in viol:
            events += 1
            types[v.get("type", "?")] += 1
            if float(v.get("timecode_sec", 0) or 0) == 0.0:
                zero_tc += 1

    print(f"\n===== {split}: {len(data)} сэмплов =====")
    print(f"  <image> != кол-ву картинок : {mismatch}")
    print(f"  сэмплов без картинок        : {no_imgs}")
    print(f"  картинок нет на диске       : {missing}")
    print(f"  макс. кадров в сэмпле       : {max_imgs}  (~{max_imgs * TOKENS_PER_IMG} токенов на картинки)")
    print(f"  ответ gpt не JSON           : {bad_json}")
    print(f"  с нарушением / без          : {positives} / {len(data) - positives}")
    print(f"  нарушений с timecode = 0.0  : {zero_tc} из {events}")
    print("  типы нарушений:")
    for t, n in types.most_common():
        print(f"    {n:4d}  {t}")

    if mismatch or no_imgs or missing or bad_json:
        ok = False
    if events and zero_tc / events > 0.5:
        print("  ⚠️ Большинство таймкодов = 0.0 — модель научится всегда выдавать 0.0. Надо чинить генерацию датасета.")
    if max_imgs * TOKENS_PER_IMG + 800 > 4096:
        print("  ⚠️ Кадры не влезут в cutoff_len=4096 — уменьши image_max_pixels в шаге 7.")

sample_img = os.path.join(DATA_DIR, data[0]["images"][0])
print("\nРазмер кадра:", Image.open(sample_img).size)
print("\n✅ Датасет в порядке" if ok else "\n❌ Есть ошибки — дальше не запускать")

## 6. Регистрация датасета

In [ ]:
import json

DATA_DIR = "/content/LLaMA-Factory/data"
info_path = f"{DATA_DIR}/dataset_info.json"
with open(info_path, encoding="utf-8") as f:
    info = json.load(f)

for split in ["train", "val"]:
    info[f"cashguard_{split}"] = {
        "file_name": f"cashguard_{split}.json",
        "formatting": "sharegpt",
        "columns": {"messages": "conversations", "images": "images"},
        "tags": {"role_tag": "from", "content_tag": "value",
                 "user_tag": "human", "assistant_tag": "gpt", "system_tag": "system"},
    }
info.pop("cashguard_train_old", None)

with open(info_path, "w", encoding="utf-8") as f:
    json.dump(info, f, ensure_ascii=False, indent=2)
print("✔ cashguard_train / cashguard_val зарегистрированы (с колонкой images)")

## 7. Конфиг обучения
Выход пишется на Google Drive (если подключён) — при обрыве сессии чекпоинты не пропадут, а повторный запуск шага 8 продолжит с последнего.

In [ ]:
import os, torch

OUTPUT_DIR = ("/content/drive/MyDrive/cashguard/lora_output"
              if os.path.isdir("/content/drive/MyDrive") else "/content/lora_output")
PRECISION = "bf16: true" if torch.cuda.get_device_capability()[0] >= 8 else "fp16: true"  # T4 → fp16

config = f"""
### Модель
model_name_or_path: Qwen/Qwen2.5-VL-7B-Instruct
trust_remote_code: true
image_max_pixels: 262144

### Метод (QLoRA)
stage: sft
do_train: true
finetuning_type: lora
quantization_bit: 4
upcast_layernorm: true
lora_rank: 16
lora_alpha: 32
lora_dropout: 0.05
lora_target: all
freeze_vision_tower: true

### Датасет
dataset: cashguard_train
eval_dataset: cashguard_val
template: qwen2_vl
cutoff_len: 4096
overwrite_cache: true
preprocessing_num_workers: 2

### Обучение
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 1.0e-4
lr_scheduler_type: cosine
warmup_steps: 10
num_train_epochs: 3
weight_decay: 0.01
max_grad_norm: 1.0
{PRECISION}
gradient_checkpointing: true
flash_attn: auto

### Логи, eval, сохранение
logging_steps: 5
eval_strategy: steps
eval_steps: 40
per_device_eval_batch_size: 1
save_steps: 10
save_total_limit: 3
output_dir: {OUTPUT_DIR}
report_to: none
seed: 42
"""

with open("/content/LLaMA-Factory/train_cashguard.yaml", "w", encoding="utf-8") as f:
    f.write(config.strip())
print("✔ Конфиг сохранён |", PRECISION, "| выход:", OUTPUT_DIR)

## 8. Обучение
Если сессия оборвалась — прогони шаги 2, 3, 4, 4b, 6, 7 и снова этот: обучение продолжится с последнего чекпоинта на Drive.

In [ ]:
%cd /content/LLaMA-Factory
!llamafactory-cli train /content/LLaMA-Factory/train_cashguard.yaml

## 9. Забрать готовый адаптер
Берём только финальный LoRA-адаптер (~сотни МБ), без промежуточных чекпоинтов. Мёржить с базовой моделью будем на GPU-сервере — в колабе на это не хватит ни RAM, ни места на Drive.

In [ ]:
import os, glob, shutil

OUTPUT_DIR = ("/content/drive/MyDrive/cashguard/lora_output"
              if os.path.isdir("/content/drive/MyDrive") else "/content/lora_output")
FINAL = "/content/cashguard_lora_final"

shutil.rmtree(FINAL, ignore_errors=True)
os.makedirs(FINAL)
for f in glob.glob(f"{OUTPUT_DIR}/*"):
    if os.path.isfile(f):
        shutil.copy(f, FINAL)
print("Файлы адаптера:", sorted(os.listdir(FINAL)))

archive = shutil.make_archive(FINAL, "zip", FINAL)
print("✔ Архив:", archive, f"({os.path.getsize(archive) / 1e6:.0f} МБ)")

if os.path.isdir("/content/drive/MyDrive"):
    shutil.copy(archive, "/content/drive/MyDrive/cashguard/")
    print("✔ Копия на Drive: MyDrive/cashguard/cashguard_lora_final.zip")
else:
    from google.colab import files
    files.download(archive)